# ASL 29-Class Testing Metrics Notebook

This notebook loads the 29-class ASL dataset split, restores the fine-tuned MobileNetV2 29-class checkpoint, and computes the evaluation metrics requested:
- F1 per class
- Micro F1
- Macro F1
- Confusion matrix
- Accuracy
- Loss

Checkpoint: `best_mobilenet_v2_finetuned_29.pth`
Path: `/kaggle/input/models/francoolanomelo/asl-trained-models/pytorch/default/3/best_mobilenet_v2_finetuned_29.pth`

In [ ]:
import json
import os
import subprocess
import sys
from collections import OrderedDict
from pathlib import Path

def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except ImportError:
        package_name = pip_name or import_name
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

for import_name, pip_name in [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('torch', 'torch'),
    ('torchvision', 'torchvision'),
    ('PIL', 'Pillow'),
]:
    ensure_package(import_name, pip_name)

import numpy as np
import pandas as pd
import torch
from IPython.display import display

def discover_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/kaggle/working')]
    for candidate in candidates:
        if (candidate / 'engine').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return cwd

REPO_URL = 'https://github.com/FrancOlano/ASL-Recognition-DL'
REPO_NAME = 'ASL-Recognition-DL'
REPO_ROOT = discover_repo_root()

if not (REPO_ROOT / 'engine').exists():
    target_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd().resolve()
    repo_dir = target_root / REPO_NAME
    if not repo_dir.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(repo_dir)])
    REPO_ROOT = repo_dir

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine import config as cfg
from engine.dataset import get_data_loaders
from engine.model_factory import build_model

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(cfg.SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Repository root: {REPO_ROOT}')
print(f'Device: {device}')

In [ ]:
def resolve_dataset_root():
    kaggle_roots = [
        Path('/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train'),
        Path('/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train'),
        Path('/kaggle/input/datasets/grassknoted/asl-alphabet'),
        Path('/kaggle/input/asl-alphabet'),
        Path('/kaggle/input/asl_alphabet'),
        Path('/kaggle/input/grassknoted-asl-alphabet'),
        Path('/kaggle/input/grassknoted/asl-alphabet'),
    ]
    for root in kaggle_roots:
        if root.exists():
            return root

    local_root = REPO_ROOT / 'data' / 'processed'
    if local_root.exists():
        return local_root

    raise FileNotFoundError(
        'Could not find a dataset root. Expected the Kaggle train split at '
        '/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train '
        'or a local data/processed folder.'
    )

dataset_root = resolve_dataset_root()

# Setting up for 29 classes (classes_to_keep=None loads all classes in data_dir, which is 29)
cfg.KAGGLE = Path('/kaggle/working').exists()
cfg.PROJECT_ROOT = REPO_ROOT
cfg.DATA_DIR = dataset_root
cfg.NUM_CLASSES = 29
cfg.MODEL_OUTPUT_DIR = REPO_ROOT / 'models' / 'checkpoints'
cfg.RESULTS_DIR = REPO_ROOT / 'results'

cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset root: {dataset_root}')

In [ ]:
train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = get_data_loaders(
    data_dir=cfg.DATA_DIR,
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS,
    classes_to_keep=None,
)

class_names = train_loader.dataset.subset.dataset.classes
num_classes = len(class_names)
cfg.NUM_CLASSES = num_classes

(cfg.RESULTS_DIR / 'classes_29.json').write_text(json.dumps(class_names, indent=2))

print('Split sizes:')
print(f'  Train: {len(train_dataset)}')
print(f'  Validation: {len(val_dataset)}')
print(f'  Test: {len(test_dataset)}')
print(f'  Number of classes: {num_classes}')
print(f'  Class names: {class_names}')

In [ ]:
RESULTS_ROOT = REPO_ROOT / 'results' / 'testing_metrics_29'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CONFUSION_DIR = RESULTS_ROOT / 'confusion_matrices'
CONFUSION_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH_KAGGLE = '/kaggle/input/models/francoolanomelo/asl-trained-models/pytorch/default/3/best_mobilenet_v2_finetuned_29.pth'

def _extract_state_dict(raw_checkpoint):
    if isinstance(raw_checkpoint, dict):
        for key in ('state_dict', 'model_state_dict', 'model', 'net'):
            value = raw_checkpoint.get(key)
            if isinstance(value, dict):
                return value
    return raw_checkpoint

def _strip_module_prefix(state_dict):
    stripped = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            stripped[key[len('module.'):]] = value
        else:
            stripped[key] = value
    return stripped

def find_checkpoint_path():
    candidates = [
        Path(CHECKPOINT_PATH_KAGGLE),
        REPO_ROOT / 'models' / 'checkpoints' / 'best_mobilenet_v2_finetuned_29.pth',
        REPO_ROOT / 'checkpoints' / 'best_mobilenet_v2_finetuned_29.pth',
        Path('best_mobilenet_v2_finetuned_29.pth')
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate

    search_roots = [
        REPO_ROOT / 'models' / 'checkpoints',
        REPO_ROOT / 'checkpoints',
        REPO_ROOT,
        Path('/kaggle/input/models'),
        Path('/kaggle/input'),
        Path('/kaggle/working'),
    ]
    for root in search_roots:
        if root.exists():
            direct_path = root / 'best_mobilenet_v2_finetuned_29.pth'
            if direct_path.is_file():
                return direct_path
            # Also check the nested kaggle path pattern
            nested_path = root / 'francoolanomelo/asl-trained-models/pytorch/default/3/best_mobilenet_v2_finetuned_29.pth'
            if nested_path.is_file():
                return nested_path

    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob('best_mobilenet_v2_finetuned_29.pth'))
    matches = sorted({path.resolve() for path in matches if path.is_file()})
    if matches:
        return matches[0]

    raise FileNotFoundError(
        "Could not resolve checkpoint best_mobilenet_v2_finetuned_29.pth. "
        "Searched paths include models/checkpoints and Kaggle inputs."
    )

def confusion_matrix_from_labels(y_true, y_pred, class_count):
    matrix = np.zeros((class_count, class_count), dtype=np.int64)
    for true_label, predicted_label in zip(y_true, y_pred):
        matrix[int(true_label), int(predicted_label)] += 1
    return matrix

def compute_metrics_from_confusion_matrix(matrix):
    true_positive = np.diag(matrix).astype(np.float64)
    predicted_total = matrix.sum(axis=0).astype(np.float64)
    actual_total = matrix.sum(axis=1).astype(np.float64)

    precision = np.divide(
        true_positive,
        predicted_total,
        out=np.zeros_like(true_positive),
        where=predicted_total != 0,
    )
    recall = np.divide(
        true_positive,
        actual_total,
        out=np.zeros_like(true_positive),
        where=actual_total != 0,
    )
    f1_scores = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) != 0,
    )

    total_true_positive = true_positive.sum()
    total_predictions = predicted_total.sum()
    total_actual = actual_total.sum()
    micro_precision = total_true_positive / total_predictions if total_predictions else 0.0
    micro_recall = total_true_positive / total_actual if total_actual else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) != 0
        else 0.0
    )

    return f1_scores, micro_f1, float(np.mean(f1_scores))

print('Helpers for loading model and computing metrics are ready.')

In [ ]:
checkpoint_path = find_checkpoint_path()
print(f'Loading checkpoint from: {checkpoint_path}')

model = build_model(
    model_type='mobilenet_v2',
    num_classes=num_classes,
    pretrained=False
).to(device)

raw_checkpoint = torch.load(checkpoint_path, map_location=device)
state_dict = _strip_module_prefix(_extract_state_dict(raw_checkpoint))
model.load_state_dict(state_dict, strict=True)
model.eval()

print("Model loaded and configured in evaluation mode.")

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

print('Evaluating model on the test loader...')
total_loss = 0.0
total_samples = 0
all_targets = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        
        predictions = torch.argmax(outputs, dim=1)
        all_targets.append(labels.cpu().numpy())
        all_predictions.append(predictions.cpu().numpy())

y_true = np.concatenate(all_targets)
y_pred = np.concatenate(all_predictions)

matrix = confusion_matrix_from_labels(y_true, y_pred, num_classes)
per_class_f1, micro_f1, macro_f1 = compute_metrics_from_confusion_matrix(matrix)
accuracy = float((y_true == y_pred).mean())
average_loss = total_loss / total_samples if total_samples else 0.0

model_confusion_csv = CONFUSION_DIR / 'mobilenet_v2_finetuned_29_confusion_matrix.csv'
pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(model_confusion_csv)

evaluation_rows = [{
    'model': 'MobileNetV2 Fine-Tuned (29 Classes)',
    'architecture': 'mobilenet_v2',
    'checkpoint': str(checkpoint_path),
    'loss': average_loss,
    'accuracy': accuracy,
    'micro_f1': micro_f1,
    'macro_f1': macro_f1,
    'confusion_matrix_path': str(model_confusion_csv),
}]

per_class_rows = []
for class_name, f1_score in zip(class_names, per_class_f1):
    per_class_rows.append({
        'class_name': class_name,
        'f1_score': float(f1_score),
    })

summary_df = pd.DataFrame(evaluation_rows)
per_class_df = pd.DataFrame(per_class_rows)

summary_csv = RESULTS_ROOT / 'summary_metrics_29.csv'
per_class_csv = RESULTS_ROOT / 'per_class_f1_29.csv'
confusions_json = RESULTS_ROOT / 'confusion_matrix_29.json'

summary_df.to_csv(summary_csv, index=False)
per_class_df.to_csv(per_class_csv, index=False)
confusions_json.write_text(json.dumps({ 'mobilenet_v2_finetuned_29': matrix.tolist() }, indent=2))

print('\nEvaluation Results:')
print(f'Loss: {average_loss:.6f}')
print(f'Accuracy: {accuracy:.6f}')
print(f'Micro F1: {micro_f1:.6f}')
print(f'Macro F1: {macro_f1:.6f}')
print(f'Confusion matrix saved to: {model_confusion_csv}')
print(f'Summary metrics saved to: {summary_csv}')
print(f'Per-class F1 metrics saved to: {per_class_csv}')
print(f'Confusion matrix JSON saved to: {confusions_json}')

print('\nSummary Metrics Table:')
display(summary_df.round(6))

print('\nPer-Class F1 Table:')
display(per_class_df.round(6))